# Skymap Realizations

This notebook shows how to generate one HEALPix skymap realization from the semi-analytic population model and visualize one PTA frequency bin.

## Imports and Setup

Skymap generation needs a HEALPix backend. If you installed the `viz` extra, `fastropop` can use `jax-healpy` or `healpy` for the map construction and `healpy` for plotting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp

import fastropop
from fastropop import SemiAnalyticPopulation


In [ ]:
population_params = {
    "n0": 10**-90.4153,
    "alphaM": -1.3800,
    "Mstar": 10**8.8272 * fastropop.MsunMKS,
    "betaz": -0.1711,
    "z0": 4.70,
}

sampling_grids = {
    "Mgrid": jnp.geomspace(fastropop.Mmin / fastropop.kg, fastropop.Mmax / fastropop.kg, 300),
    "zgrid": jnp.linspace(fastropop.zmin, fastropop.zmax, 300),
    "fgrid": jnp.geomspace(fastropop.fminNG15 * fastropop.s, fastropop.fmaxNG15 * fastropop.s, 300),
}

pop = SemiAnalyticPopulation(
    population_params=population_params,
    sampling_grids=sampling_grids,
)


## Choose a Manageable Number of Binaries

The full expected number of binaries can be very large, so for a tutorial skymap it is often practical to draw one realization and then cap the number of binaries used in the map generation step.

In [ ]:
Nbinaries_mean = pop.compute_Nbinaries(verbose=True)
Nbinaries_draw = int(pop.generate_poisson_realization(1, Nbinaries_mean=Nbinaries_mean, key=0)[0])
Nbinaries_skymap = min(Nbinaries_draw, 100000)

Nbinaries_mean, Nbinaries_draw, Nbinaries_skymap


## Generate the Skymaps

The skymap generator returns three arrays: the stacked total map container and the separate plus and cross polarization maps.

In [ ]:
Nside = 16
skymaps_tot, skymaps_plus, skymaps_cross = pop.generate_skymaps(
    Nbinaries=Nbinaries_skymap,
    Nside=Nside,
    batch_size=100000,
    key=1,
    verbose=True,
)

skymaps_tot.shape, skymaps_plus.shape, skymaps_cross.shape


## Plot One Frequency Bin

Here we show the first PTA frequency bin as a Mollweide projection. The plotting helper can also show `plus` or `cross` instead of the combined `total` amplitude.

In [ ]:
fastropop.plot_skymap(
    skymaps_tot,
    freq_index=0,
    polarization="total",
    log_scale=True,
    title="Total skymap, first PTA frequency bin",
)
